# Circuit Tracing

Before Spring break, we studied sparse Autoencoders (SAE's), which are a dictionary learning technique for isolating human interpretable features in LLMs. 

**features** are building blocks the model uses in its computations. But we also want to be able o understand **circuits**, groups of features which interact to produce model outputs

**SAE's cannot isolate circuits, they are intended for feature discovery**. This motivations an exploration of **Circuit Tracing** a technique developed in the last year for isolating circuits within language models which perform human-interpretable tasks. 

We'll start by reviewing SAE's. In particular I will introduce you to a new nonlinear activation method called JumpRELU, which performs better in SAE's and which is used in Circuit Tracing. Then we will describe circuit tracing methods and write code for it

- Sparse Autoencoders are a dictionary learning technique pioneered by Anthropic for extracting monosemantic, interpretable features from neural network activations
- They're trained on the internal activations of the MLP layer or the residual stream
- They consist of an encoder and decoder, each a single linear layer, with a ReLU activation between them:
$$f(x) = \operatorname{ReLU}(W_{enc}x + b_{enc}), \quad \hat{x} = W_{dec}z + b_{dec}$$
- The SAE learns an **overcomplete basis** ($d_{SAE} \gg d_{model}$) in the columns of $W_{dec}$ and is trained to reconstruct activations $h$ as $\hat{h}$.
- A composite loss function 

$$\mathcal{L} = \mathcal{L}_{\text{reconstruction}} + \mathcal{L}_{\text{sparsity}}$$

is learned, where 

$$  \mathcal{L}_{\text{reconstruction}}  = \frac{1}{|X|} \sum_{\mathbf{x} \in X} ||\mathbf{x} - \hat{\mathbf{x}}||_2^2 $$

and $\mathcal{L}_{\text{sparsity}}$ is given by the weighted sum of the L2 norm of the decoder weights $W_{dec}$ scaled by the activations in the SAE's latent space:

$$\mathcal{L}_{\text{sparsity}} = \lambda \sum_i |\mathbf{f}_i(x)| \, ||W_{d,i}||_2 $$

- The **column vectors of the decoder** $W_{dec}$ are the learned feature directions in activation space — these are the monosemantic, interpretable features. The encoder learns to detect when each feature is present


<img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/sae-diagram-2.png" width="900">

## (Topics we didn't get to last week) Feature Suppression

Feature Supression is a consequence of the sparisty penalty in the SAE loss funftion which causese the model to push for smaller activation values $f(x)$ in its latent space. As I'll demonstrate with a small-scale theoretical example below, this suppression of activations results in the model suppressing features below a certain activation threshold, leading to worse reconstruction loss. 

### Demo of Why Feature Suppression Occurs

Let's say we want to train an SAE to solve the simplest possible case: there is only one binary feature in one dimension: $x=1$ with probability $p$ and $x=0$ with probability $(1-p)$. The optimal SAE would extract feature activations $f(x)=1$ and have a decoder weiht $W_d = 1$. If we train the standard SAE architecture described above 
$$f(x) = \operatorname{ReLU}(W_{enc}x + b_{enc})$$
$$y = W_{dec}z + b_{dec}$$

And use the same loss function described above: 
$$\mathcal{L(x, f(x), y)} = \frac{1}{|X|=1} \sum_{\mathbf{x} \in X} ||\mathbf{x} - y||_2^2 + \lambda \sum_i |\mathbf{f}_i(x)| \, ||W_{d,i}||_2 $$

The optimal outputs for the model 

activations for the model could learn

$$y^* = \argmin_y (\mathcal{L})$$

We want $ y^* = x = 1$. Let's plug into our equation and see if that is actually the case. Remember that $X=1$ because we have just 1 feature we want to learn. $x=1$ because the loss is zero when $x=0$. Remember we're taking the expectation over the values $x$ can take on. Since $W_d = 1$ (without loss of generality, since the sparsity penalty $\lambda |f(x)| \cdot ||W_d||$ penalizes the product of activation magnitude and decoder norm), the reconstruction is $y = W_d \cdot a = a$, so:

$$\mathcal{L}(a) = p\left((a - 1)^2 + \lambda |a| \cdot 1\right) + (1-p)\left((0-0)^2 + \lambda |0| \cdot 1\right)$$

The second term vanishes entirely, and since $p$ is a positive constant it doesn't affect the argmin, so:

$$= \arg\min_a (a - 1)^2 + \lambda a$$

Expanding:

$$= \arg\min_a \; a^2 - 2a + 1 + \lambda a$$

$$= \arg\min_a \; a^2 + (\lambda - 2)a + 1$$

Taking the derivative and setting to zero:

$$\frac{d\mathcal{L}}{da} = 2a + (\lambda - 2) = 0$$

$$\boxed{a^* = 1 - \frac{\lambda}{2}}$$

Since $\lambda > 0$, we have $a^* < 1$. The SAE learns to output activations that are **smaller** than the true value, accepting reconstruction error $(x - \hat{x})^2 = (\lambda/2)^2 > 0$ in order to pay a lower sparsity penalty. This is feature suppression — it is not a training failure but an inherent property of the L1 loss formulation.


## (Topics we didn't get to last week) JumpRELU


One way of correcting for feature expression is using the JumpRELU loss function. 

The JumpReLU architecture is identical to regular SAEs, except we have an extra parameter $\theta$ (which is a vector of length `d_sae` representing the threshold for each latent), and our activation function is $\operatorname{JumpReLU}_\theta(z) = z H(z - \theta)$, where $z$ are the pre-activation SAE hidden values and $H$ is the Heaviside step function (i.e. value of 1 if $z > \theta$ and 0 otherwise). The function looks like:

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/wZqqQysfLrt2CFx4T/zzrdot3xexvcz3mqghn8" width="240">

We train JumpReLU SAEs against the following loss function:

$$
\mathcal{L}(\mathbf{x}):=\underbrace{\|\mathbf{x}-\hat{\mathbf{x}}(\mathbf{f}(\mathbf{x}))\|_2^2}_{\mathcal{L}_{\text {reconstruct }}}+\underbrace{\lambda\|\mathbf{f}(\mathbf{x})\|_0}_{\mathcal{L}_{\text {sparsity }}}
$$

This is just like the standard SAE loss function, except we penalize the L0 norm of the hidden activations directly, rather than L1. The question remains - how do we backprop these terms wrt $\theta$, since the heaviside function and L0 norm are both discontinuous? The answer comes from **straight-through-estimators** (STEs), which are a method for approximating gradients of non-differentiable functions. Specifically, we first rewrite the L0 term in terms of the Heaviside step function  $\|\mathbf{f}(\mathbf{x})\|_0 = \sum_{i=1}^{d_{\text{sae}}} H(\pi_i(\mathbf{x}) - \theta_i)$ where $\pi_i(\mathbf{x})$ are the pre-JumpReLU SAE hidden values. Next, since we've reduced the problem to just thinking about the Heaviside and JumpReLU functions, we can use the following estimates:

$$
\begin{aligned}
\frac{ð}{ð \theta} \operatorname{JumpReLU}_\theta(z) & :=-\frac{\theta}{\varepsilon} K\left(\frac{z-\theta}{\varepsilon}\right) \\
\frac{ð}{ð \theta} H(z-\theta) & :=-\frac{1}{\varepsilon} K\left(\frac{z-\theta}{\varepsilon}\right)
\end{aligned}
$$

where $K$ is some **valid kernel function** (i.e. must satisfy the properties of a centered, finite-variance probability density function). In the GDM experiments, they use the **rectangle function** $H(z+\frac{1}{2}) - H(z-\frac{1}{2})$.

We provide 2 intuitions for why this works below - one functional/visual, and one probability-based. If you really don't care about this, you can skip to the exercise section (although we do encourage you to read at least one of these).

#### Functional / visual intuition

What we're effectively doing here is approximating discontinuous functions with sharp cumulative distribution functions. For example, take the heaviside function $H(z) = \mathbf{1}(z > 0)$. We can approximate this with a cdf $F$ which is sharp around the discontinuity (i.e. $F(z) = 0$ for all slightly negative $z$, and $F(z) = 1$ for all slightly positive $z$). The reason our derivative approximations above involve probability density functions $K$ is that the derivative of a cumulative distribution function $F$ is its probability density function.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/jumprelu-1.png" width="560">

If you're interested, the dropdown below derives this result using actual calculus (i.e. showing that the integral of these approximate derivatives over a sufficiently large region equals the size of the jump discontinuity). Note that this isn't crucial and we don't necessarily recommend it unless you're especially curious.

Derivation of this integral result (less important)

Suppose $F$ is the cumulative distribution function of $K$, so we have $F'(z) = K(z)$ and $F(-\infty) = 0, F(\infty) = 1$. Then let's compute the integral of the approximated Heaviside function over a region with centre $z$ and radius $\epsilon C$. Note we're computing the integral over a negative range, because it's moving $\theta$ from above $z$ to below $z$ that causes the output to jump from 0 to 1.

$$
\int\limits_{z+\epsilon C}^{z-\epsilon C} -\frac{1}{\epsilon} K\left(\frac{z-\theta}{\epsilon}\right) d\theta = \int\limits_{-C}^{C} K(\theta)\; d\theta = F(C) - F(-C) \xrightarrow[C \to \infty]{} 1 - 0 = 1
$$

which is the size of the jump discontinuity. Note that for our choice of the rectangle function $H(z+\frac{1}{2}) - H(z-\frac{1}{2})$ as the kernel function, this result holds even when we integrate over the small region with $C=\frac{1}{2}$, i.e. $\theta \in [z - \frac{\epsilon}{2}, z + \frac{\epsilon}{2}]$. It makes sense that we'd want a property like this, because the effect on our $\theta$ values should be largest when we're close to the jump discontinuity, and zero in most other regions.

For the JumpReLU term, after applying the reparametrization above, we can recognize the integral of $\theta K(\theta)$ as being the expected value of a variable with pdf $K$ (which is zero by our choice of $K$**), meaning we get:

$$
\int\limits_{z+\epsilon C}^{z-\epsilon C} -\frac{\theta}{\epsilon} K\left(\frac{z-\theta}{\epsilon}\right) d\theta = \int\limits_{-C}^{C} (z - \theta) K(\theta)\; d\theta = \int\limits_{-C}^{C} z K(\theta)\; d\theta \xrightarrow[C \to \infty]{} z
$$

which once again equals the size of the jump discontinuity, and once again is also a result that holds if we just take the region $\theta \in [z - \frac{\epsilon}{2}, z + \frac{\epsilon}{2}]$ for our chosen kernel $K$.

**Technically it's only zero if we integrate over the entire domain. But our choice of $K$ (as well as most reasonable choices for $K$) are not only centered at zero but also symmetric around zero and decay rapidly as we move away from zero, meaning we can make this assumption.



#### Probability-based intuition

Another way to think about this is that our inputs $x$ have some element of randomness. So our loss function values $\mathcal{L}_\theta(x)$ are themselves random variables which approximate the expected loss $\mathbb{E}_x\left[\mathcal{L}_\theta(x)\right]$. And it turns out that even if we can't compute the gradient of the loss directly if the loss contains a non-continuous term, we can compute the gradient of the expected loss. For example, consider the sparsity term $\|\mathbf{f}(\mathbf{x})\|_0 = \sum_{i=1}^{d_{\text{sae}}} H(z_i - \theta_i)$ (where $z_i$ are the pre-JumpReLU hidden values). This is not differentiable at zero, but its expected value is $\mathbb{E}_x \|\mathbf{f}(\mathbf{x})\|_0 = \sum_{i=1}^{d_{\text{sae}}} \mathbb{P}(z_i > \theta_i)$ which is differentiable - the derivative wrt $\theta_i$ is $-\mathbb{E}_x\left[p_i(z_i-\theta_i)\right]$, where $p_i$ are the probability density functions for $z_i$.

Okay, so we know what we want our derivatives to be in expectation, but why does our choice $\frac{ð}{ð \theta} H(z-\theta) :=-\frac{1}{\varepsilon} K\left(\frac{z-\theta}{\varepsilon}\right)$ satisfy this? The answer is that this expression is a form of **kernel density estimation** (KDE), i.e. it approximates the pdf for a variable by smoothing out its empirical distribution.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/jumprelu-2.png" width="620">

## JumpRELU in Code

## Circuit Tracing